# T03 — Upsampler Wrapper Check

Checks `anyup/upsamplers.py` (task T03, see `docs/plan/P1_experiments/T03_upsampler_wrapper.md`) with the real weights.

1. Safe checkpoint load (`weights_only=True` + `TStage` stub) of the step6500 checkpoint: keys, `TStage` objects, strict load
2. AnyUp3D wrapper vs Exp 2 `run_anyup3d` (exact) and AnyUp2D wrapper vs Exp 2 `run_anyup2d` (exact), VideoMAE, first 16 frames of `bear`
3. Bilinear wrapper vs Exp 3 `upsample_bilinear` (exact), Swin-T per-frame, `PROP_SIZE=56`
4. Full path on all 82 frames: `iter_backbone_chunks` + each upsampler (shapes, indices, time, peak memory per chunk)
5. AnyUp3D `q_chunk_size` vs None (INFO)

GPU runtime; **A100 recommended** (AnyUp3D at 224×224 with `q_chunk_size=None` needs a lot of memory, as in Exp 2).
Run all cells, then paste the **final summary cell's output** back.

In [ ]:
# ── Cell 1 · Repo + Drive ───────────────────────────────────────────────────
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-task02"   # branch that contains anyup/upsamplers.py (T03 continues on the T02 branch)
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Imports + config ───────────────────────────────────────────────
import json, time
from pathlib import Path
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image

from anyup.backbones import BackboneLoader
from anyup.video import open_video, iter_backbone_chunks
from anyup.upsamplers import UpsamplerLoader, load_anyup3d_checkpoint, ANYUP3D_ARCH
from anyup.model import AnyUp
from anyup.model_2d import AnyUp2D

DEVICE       = "cuda"
device       = torch.device(DEVICE)   # name used by the copied Exp 2 functions
CKPT_ANYUP3D = "/content/drive/MyDrive/results/anyup3d_step6500.pth"   # regression checkpoint (T03 Q11), as Exp 2/3
ANYUP2D_URL  = "https://github.com/wimmerth/anyup/releases/download/checkpoint/anyup_paper.pth"
DAVIS_URL    = "https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip"
DAVIS_ZIP    = "/content/DAVIS-2017-trainval-480p.zip"
DAVIS_ROOT   = "/content/DAVIS"
CLIP         = "bear"   # 82 frames, 854×480 (DAVIS-2017 train)
N_REG        = 16       # ↓ frames for the exact regression checks (one VideoMAE chunk → 8 feature frames); fixed by VideoMAE
PROP_SIZE    = 56       # ↓ Exp 3 propagation / bilinear output size (Exp 3 Cell 2)
Q_CHUNK      = 4096     # ↓ AnyUp3D query chunk for the q_chunk check (Cell 8); smaller = less memory, slower
REL_TOL, MIN_COS = 1e-3, 0.99999   # INFO thresholds for the q_chunk check (T02 Q22/Q25 convention)
RESULTS_JSON = "/content/drive/MyDrive/results/T03_upsampler_check.json"   # results saved here (Cell 9)

RESULTS = []   # (check name, passed: bool | None for informational, detail str)
def record(name, passed, detail):
    RESULTS.append((name, passed, detail))
    tag = "INFO" if passed is None else ("PASS" if passed else "FAIL")
    print(f"[{tag}] {name}: {detail}")

def compare_rel(a, b):
    """→ (max_abs, relative error max_abs / max|a|, worst-frame cosine). a, b: (T, ...)"""
    a, b = a.float().cpu(), b.float().cpu()
    max_abs = (a - b).abs().max().item()
    cos = F.cosine_similarity(a.flatten(1), b.flatten(1), dim=1).min().item()
    return max_abs, max_abs / a.abs().max().item(), cos

def peak_mib():
    return torch.cuda.max_memory_allocated() / 2**20

print("Device:", DEVICE, torch.cuda.get_device_name(0), "| upsamplers:", UpsamplerLoader.available())

In [ ]:
# ── Cell 3 · DAVIS (always downloaded from ETH) + frames ───────────────────
if not os.path.exists(os.path.join(DAVIS_ROOT, "ImageSets")):
    if not os.path.exists(DAVIS_ZIP):
        subprocess.run(["wget", "-q", "-O", DAVIS_ZIP, DAVIS_URL], check=True)
    subprocess.run(["unzip", "-q", DAVIS_ZIP, "-d", "/content/"], check=True)

CLIP_DIR = Path(DAVIS_ROOT) / "JPEGImages/480p" / CLIP
SRC      = open_video(CLIP_DIR)
N        = len(SRC)
FRAMES16 = SRC.get(range(N_REG))   # (16, 3, 480, 854) float [0, 1]
print(f"clip={CLIP} frames={N} native_size={SRC.native_size} regression frames={tuple(FRAMES16.shape)}")

## 1 · Safe checkpoint load

In [ ]:
# ── Cell 4 · weights_only=True + TStage stub on the real checkpoint ─────────
# No notebook-side TStage is defined: the allowlisted stub lives in anyup.upsamplers.
try:
    CKPT = load_anyup3d_checkpoint(CKPT_ANYUP3D)
    err = None
except Exception as e:
    CKPT, err = None, f"{type(e).__name__}: {e}"
record("safe load (weights_only=True + TStage)", err is None, f"keys={sorted(CKPT) if CKPT else None} error={err}")

if CKPT is not None:
    cfg = CKPT.get("cfg", {})
    stages = cfg.get("t_schedule", [])
    record("checkpoint content", None,
           f"global_step={CKPT.get('global_step')} data_epoch={CKPT.get('data_epoch')} "
           f"cfg.window_t={cfg.get('window_t')} cfg.t_k={cfg.get('t_k')} cfg.kernel_size_lfu={cfg.get('kernel_size_lfu')} "
           f"t_schedule={stages}")
    arch_cfg = {k: cfg[k] for k in ANYUP3D_ARCH if k in cfg}
    mism = {k: (v, ANYUP3D_ARCH[k]) for k, v in arch_cfg.items() if v != ANYUP3D_ARCH[k]}
    record("ANYUP3D_ARCH == checkpoint cfg (keys present in cfg)", not mism,
           f"compared={sorted(arch_cfg)} mismatches(cfg, default)={mism}")

In [ ]:
# ── Cell 5 · Wrappers + Exp 2 reference models ─────────────────────────────
up3d = UpsamplerLoader.load("anyup3d", ckpt_path=CKPT_ANYUP3D, device=DEVICE)   # strict=True inside
up2d = UpsamplerLoader.load("anyup2d", device=DEVICE)
upbl = UpsamplerLoader.load("bilinear", device=DEVICE)

# Exp 2 Cell 1, verbatim constructor args; the state dict comes from the safe loader
# (Exp 2 used torch.load(weights_only=False), which we no longer run)
model_3d = AnyUp(
    input_dim       = 3,
    qk_dim          = 128,
    kernel_size     = 1,
    kernel_size_lfu = 5,
    window_ratio    = 0.1,
    num_heads       = 4,
    t_k             = 3,
    window_t        = 4,
).to(device).eval()
res3 = model_3d.load_state_dict(CKPT["anyup"], strict=True)
record("Exp 2 AnyUp3D strict load", not res3.missing_keys and not res3.unexpected_keys,
       f"missing={len(res3.missing_keys)} unexpected={len(res3.unexpected_keys)}")

model_2d = AnyUp2D().to(device).eval()
sd_2d = torch.hub.load_state_dict_from_url(ANYUP2D_URL, map_location=device, weights_only=True)
model_2d.load_state_dict(sd_2d)

def same_state(a, b):
    sa, sb = a.state_dict(), b.state_dict()
    return sa.keys() == sb.keys() and all(torch.equal(sa[k].cpu(), sb[k].cpu()) for k in sa)
record("wrapper weights == Exp 2 weights", same_state(up3d.model, model_3d) and same_state(up2d.model, model_2d),
       f"anyup3d={same_state(up3d.model, model_3d)} anyup2d={same_state(up2d.model, model_2d)} "
       f"window_t={up3d.model.window_t} ckpt_step={up3d.ckpt_step}")

## 2 · AnyUp3D / AnyUp2D vs Exp 2 (exact)

In [ ]:
# ── Cell 6 · Exp 2 run_anyup3d / run_anyup2d (Exp 2 Cells 4–5, verbatim) ───
def run_anyup3d(feats, guides):
    """
    feats  : (8, 768, 14, 14)
    guides : (8, 3, 224, 224)
    Returns: (8, 768, 224, 224)
    """
    # rearrange to (1, C, T, H, W) for the model
    feat_vol  = feats.permute(1, 0, 2, 3).unsqueeze(0).to(device)    # (1, 768, 8, 14, 14)
    guide_vol = guides.permute(1, 0, 2, 3).unsqueeze(0).to(device)   # (1, 3, 8, 224, 224)
    with torch.no_grad():
        out = model_3d(guide_vol, feat_vol)                           # guide first, features second
    return out.squeeze(0).permute(1, 0, 2, 3).cpu()                  # (8, 768, 224, 224)

def run_anyup2d(feats, guides):
    results = []
    for t in range(feats.shape[0]):        # iterate over T=8 frames independently
        feat_t  = feats[t].unsqueeze(0).to(device)    # (1, 768, 14, 14)
        guide_t = guides[t].unsqueeze(0).to(device)   # (1, 3, 224, 224)
        with torch.no_grad():
            out_t = model_2d(guide_t, feat_t)          # guide first, features second
        results.append(out_t.squeeze(0).cpu())         # (768, 224, 224)
    return torch.stack(results, dim=0)                 # (8, 768, 224, 224)

# Same inputs for both paths: one VideoMAE chunk (16 frames → 8 feature frames)
bb = BackboneLoader.load("videomae", device=DEVICE)
chunk = next(iter_backbone_chunks(bb, open_video(FRAMES16)))   # features (8, 768, 14, 14), guide (8, 3, 224, 224)
feats, guides = chunk.features.cpu(), chunk.guide.cpu()
del bb; torch.cuda.empty_cache()

for name, old_fn, up in [("anyup3d", run_anyup3d, up3d), ("anyup2d", run_anyup2d, up2d)]:
    old = old_fn(feats, guides)                     # (8, 768, 224, 224) CPU
    torch.cuda.empty_cache()
    res = up(chunk)                                 # chunk on GPU → output on GPU
    new = res.features.cpu()
    mx, rel, cs = compare_rel(old, new)
    record(f"{name} wrapper == Exp 2 run_{name}", tuple(new.shape) == tuple(old.shape) and mx == 0,
           f"shape={tuple(new.shape)} max_abs={mx:.2e} rel={rel:.2e} min_cos={cs:.7f} "
           f"meta.upsampler={res.meta['upsampler']} output_size={res.meta['output_size']}")
    del old, new, res; torch.cuda.empty_cache()

## 3 · Bilinear vs Exp 3 (exact)

In [ ]:
# ── Cell 7 · Exp 3 upsample_bilinear (Exp 3 Cell 6, verbatim) ──────────────
@torch.no_grad()
def upsample_bilinear(feats_TCHW):
    return F.interpolate(
        feats_TCHW,
        size=(PROP_SIZE, PROP_SIZE),
        mode="bilinear",
        align_corners=False,
    )

bb = BackboneLoader.load("swin_t", device=DEVICE)   # per_frame (Exp 3 frame-level features)
chunk_s = next(iter_backbone_chunks(bb, open_video(FRAMES16)))   # features (16, 768, 7, 7)
del bb; torch.cuda.empty_cache()
old = upsample_bilinear(chunk_s.features.cpu())                  # CPU, as Exp 3
cpu_chunk = type(chunk_s)(features=chunk_s.features.cpu(), guide=chunk_s.guide.cpu(), meta=chunk_s.meta)
upbl_cpu = UpsamplerLoader.load("bilinear", device="cpu")         # same device as Exp 3's F.interpolate
new = upbl_cpu(cpu_chunk, output_size=PROP_SIZE).features
mx, rel, cs = compare_rel(old, new)
record("bilinear wrapper == Exp 3 upsample_bilinear", tuple(new.shape) == tuple(old.shape) and mx == 0,
       f"shape={tuple(new.shape)} max_abs={mx:.2e}")
new_gpu = upbl(chunk_s, output_size=PROP_SIZE).features.cpu()
mx, rel, cs = compare_rel(old, new_gpu)
record("bilinear wrapper on GPU vs Exp 3 on CPU", None, f"max_abs={mx:.2e} rel={rel:.2e} min_cos={cs:.7f}")
del old, new, new_gpu

## 4 · Full path on all frames + 5 · q_chunk_size

In [ ]:
# ── Cell 8 · iter_backbone_chunks + every upsampler, all 82 frames ─────────
bb = BackboneLoader.load("videomae", device=DEVICE)
for name, up in [("bilinear", upbl), ("anyup2d", up2d), ("anyup3d", up3d)]:
    idx, shapes, times, mems = [], [], [], []
    for ch in iter_backbone_chunks(bb, SRC):
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        res = up(ch)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0); mems.append(peak_mib())
        shapes.append(tuple(res.features.shape)); idx += res.meta["frame_index"]
        del res                                         # outputs not kept (41 × 768 × 224² floats ≈ 6.3 GB)
    ok = idx == list(range(0, N, 2)) and all(s[1:] == (768, 224, 224) for s in shapes)
    record(f"full path videomae + {name}", ok,
           f"chunks={len(shapes)} t_out={[s[0] for s in shapes]} indices_ok={idx == list(range(0, N, 2))} "
           f"time_s={[round(t, 2) for t in times]} peak_MiB={[round(m) for m in mems]}")
    torch.cuda.empty_cache()

# q_chunk_size vs None, on the 16-frame chunk from Cell 6
up3d_q = UpsamplerLoader.load("anyup3d", ckpt_path=CKPT_ANYUP3D, device=DEVICE, q_chunk_size=Q_CHUNK)
out = {}
for label, up in [("None", up3d), (str(Q_CHUNK), up3d_q)]:
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    out[label] = up(chunk).features.cpu()
    torch.cuda.synchronize()
    out[label + "_stats"] = (time.perf_counter() - t0, peak_mib())
    torch.cuda.empty_cache()
mx, rel, cs = compare_rel(out["None"], out[str(Q_CHUNK)])
(tn, mn), (tq, mq) = out["None_stats"], out[f"{Q_CHUNK}_stats"]
record(f"anyup3d q_chunk_size={Q_CHUNK} vs None", None,
       f"max_abs={mx:.2e} rel={rel:.2e} min_cos={cs:.7f} within(rel<{REL_TOL}, cos>{MIN_COS})={rel < REL_TOL and cs > MIN_COS} | "
       f"None: {tn:.2f}s {mn:.0f} MiB | {Q_CHUNK}: {tq:.2f}s {mq:.0f} MiB")
del bb, out; torch.cuda.empty_cache()

## Summary — paste this output back

In [ ]:
# ── Cell 9 · Summary (also saved to Drive) ─────────────────────────────────
import transformers, timm
env = (f"torch={torch.__version__} torchvision={__import__('torchvision').__version__} "
       f"transformers={transformers.__version__} timm={timm.__version__} gpu={torch.cuda.get_device_name(0)}")
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(env); print("commit:", commit); print(f"clip={CLIP} N={N} ckpt={CKPT_ANYUP3D}")
for name, passed, detail in RESULTS:
    tag = "INFO" if passed is None else ("PASS" if passed else "FAIL")
    print(f"[{tag}] {name}: {detail}")
n_fail = sum(p is False for _, p, _ in RESULTS)
print(f"\n{len(RESULTS)} checks, {n_fail} failed")
os.makedirs(os.path.dirname(RESULTS_JSON), exist_ok=True)
with open(RESULTS_JSON, "w") as f:
    json.dump(dict(env=env, commit=commit, clip=CLIP, ckpt=CKPT_ANYUP3D,
                   results=[dict(name=n, passed=p, detail=d) for n, p, d in RESULTS]), f, indent=1)
print("saved:", RESULTS_JSON)